# Workflow Evaluation

There would be 30 prompts in a form of csv where there are these combinations

| | categories | prompt styles | stocks |
|---|---:|---:|---:|
| Combinations | 5 | 7 | 5 |
| **Total prompts** | | | **175** |



## Import Libraries

In [1]:
from collections import defaultdict
import json
from pathlib import Path
from IPython.display import clear_output

from deepeval import evaluate
from deepeval.metrics import (
    TaskCompletionMetric,       # Did the crew actually finish the job correctly (uses tools_called + actual_output)
    ArgumentCorrectnessMetric,  # Were tool call arguments correct given the user input (uses tools_called)
    AnswerRelevancyMetric,      # Is the report relevant to the user prompt (input vs actual_output)
    HallucinationMetric,        # Is the report grounded in the gathered context (context vs actual_output)
)
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval.models import OpenAIModel
import csv
import sys

PROJECT_PATH = Path("..").resolve()
sys.path.append(str(PROJECT_PATH))

# import the workflows
from utils.workflows.workflow_A import workflow_A_run as workflow_A
from utils.workflows.workflow_B import workflow_B_run as workflow_B
from utils.workflows.workflow_C import workflow_C_run as workflow_C

/home/horizontal_roterien_katze/.local/share/mamba/envs/thesis/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13811.75it/s]


In [2]:
# Load the 175-prompt test matrix
with open(PROJECT_PATH / "prompts_test_matrix_v2.csv") as f:
    prompt_rows = list(csv.DictReader(f))

print(f"{len(prompt_rows)} prompts loaded")

175 prompts loaded


## Evaluation Design

Each workflow run produces one `LLMTestCase` per prompt. The four metrics plug into
different fields of the same test case:

| Metric | Reads | What it measures here |
|---|---|---|
| `TaskCompletionMetric` | `input`, `actual_output`, `tools_called` | Was the workflow's final report a correct completion of the user's request |
| `ArgumentCorrectnessMetric` | `input`, `tools_called` | Did the agents pass sensible arguments to their tools (e.g. the stock ticker in the search query) |
| `AnswerRelevancyMetric` | `input`, `actual_output` | Does the final report actually answer the user prompt |
| `HallucinationMetric` | `context`, `actual_output` | Is every claim in the report traceable to gathered news articles |

`context` is the tool output we captured with the spy wrappers (search results +
sentiment outputs), i.e. the source material the report was written from.

In [3]:
async def run_workflow(workflow, topic: str) -> tuple[str, list[dict]]:
    """Run one workflow. Returns (report, tool_calls) via the workflow's built-in recorder."""
    report, tool_calls = await workflow(topic)
    return report, tool_calls


MAX_PARAM_CHARS = 500  # chars kept per tool-call parameter value in judge prompts
CONTEXT_BUDGET_CHARS = 240_000  # total chars across tool outputs; ~60k tokens inside the judge's 131k window

def make_tool_calls(tool_calls: list[dict], max_context_chars: int | None = 8000) -> tuple[list[ToolCall], list[str]]:
    """Convert recorded calls into deepeval ToolCall objects + grounding context strings.

    tool_calls: rows recorded by the workflow module: {tool, args, kwargs, output}
    """
    d_calls = []
    context = []
    for call in tool_calls or []:
        n_calls = max(len(tool_calls), 1)
        per_call = max_context_chars
        if max_context_chars is not None:
            per_call = min(max_context_chars, max(400, CONTEXT_BUDGET_CHARS // n_calls))
        params = dict(call.get("kwargs") or {})
        if not params:
            for i, a in enumerate(call.get("args") or []):
                params[f"arg{i}"] = a
        # judge-facing truncation: params may embed whole drafts (e.g. report-renderer
        # intermediates with 20k+ char kwargs). The judge verifies argument SENSIBILITY,
        # not full payloads; gpt-oss:120b's 131k window overflows on ~100+ call cases.
        params = {k: str(v)[:MAX_PARAM_CHARS] for k, v in params.items()}
        d_calls.append(ToolCall(name=call["tool"], description="", input_parameters=params))
        raw = call.get("output") or ""
        context.append(f"{raw[:per_call]}..." if per_call is not None else raw)
    return d_calls, context

In [4]:
def build_test_case(prompt: str, report: str, tool_calls: list[dict]) -> LLMTestCase:
    d_calls, context = make_tool_calls(tool_calls)

    return LLMTestCase(
        input=prompt,
        actual_output=report,
        context=context,
        tools_called=d_calls,
    )

In [5]:
# Choose which LLM judges the metrics.
# - "ollama": minimax-m3.1-flash via ollama cloud (uses OLLAMA_API_KEY from .env) — no OpenAI TPM limits
# - "openai": deepeval's default gpt-5.4 judge (OPENAI_API_KEY)
EVALUATOR_PROVIDER = "ollama"  # judge: DeepSeek V4.1 Flash via ollama cloud

import re, asyncio
from typing import Optional, Tuple, Union
from pydantic import BaseModel, ValidationError
from dotenv import load_dotenv

load_dotenv(PROJECT_PATH / ".env")

# --- DNS resilience -----------------------------------------------------------
# The laptop's DNS flaps across suspend/resume / network transitions ("Name or
# service not known") while long-running kernels live past them. Cache successful
# resolutions and fall back to the last-known-good IPs when the resolver blips.
import socket as _socket

_dns_cache: dict = {}
_orig_getaddrinfo = _socket.getaddrinfo

def _cached_getaddrinfo(host, *args, **kwargs):
    key = (host, args, tuple(sorted(kwargs.items())))
    try:
        result = _orig_getaddrinfo(host, *args, **kwargs)
        _dns_cache[key] = result
        return result
    except OSError:
        if key in _dns_cache:
            return _dns_cache[key]
        raise

_socket.getaddrinfo = _cached_getaddrinfo
import os
# GLM cloud calls on giant raw-string contexts exceed deepeval's per-task timeouts.
# Env is read at call time by get_settings(); safe to set here before evaluate().
os.environ.setdefault("DEEPEVAL_DISABLE_TIMEOUTS", "1")

from deepeval.models import OllamaModel

import time, random

TRANSIENT_MARKERS = (
    "429", "concurrent request slot", "timeout", "timed out",
    "connect", "502", "503", "504",
)

def _chat_with_retries(client, attempts: int = 6, base_delay: float = 2.0, **chat_kwargs):
    """Retry transient ollama-cloud errors (slot 429s, timeouts, connect errors)."""
    for attempt in range(attempts):
        try:
            return client.chat(**chat_kwargs)
        except Exception as e:
            transient = any(s in str(e).lower() for s in TRANSIENT_MARKERS)
            if not transient or attempt == attempts - 1:
                raise
            delay = base_delay * (2 ** attempt) + random.uniform(0, 1)
            print(f"[retry] transient error ({type(e).__name__}: {str(e)[:120]}), retrying in {delay:.1f}s "
                  f"(attempt {attempt + 1}/{attempts})")
            time.sleep(delay)

async def _achat_with_retries(client, attempts: int = 6, base_delay: float = 2.0, per_call_timeout: float = 300.0, **chat_kwargs):
    """Async variant of _chat_with_retries. Each attempt is hard-capped by per_call_timeout
    so a stalled cloud connection raises TimeoutError and is retried instead of hanging
    a MAX_CASE_WORKERS slot (and the whole run) forever."""
    for attempt in range(attempts):
        try:
            return await asyncio.wait_for(client.chat(**chat_kwargs), timeout=per_call_timeout)
        except Exception as e:
            transient = (isinstance(e, (TimeoutError, asyncio.TimeoutError))
                         or any(s in str(e).lower() for s in TRANSIENT_MARKERS))
            if not transient or attempt == attempts - 1:
                raise
            delay = base_delay * (2 ** attempt) + random.uniform(0, 1)
            print(f"[retry] transient error ({type(e).__name__}: {str(e)[:120]}), retrying in {delay:.1f}s "
                  f"(attempt {attempt + 1}/{attempts})")
            await asyncio.sleep(delay)




class CloudJudgeModel(OllamaModel):
    """Ollama model for cloud-proxied judges that ignore `format=` and wrap JSON in code fences.
    Strips fences / extracts the outermost JSON object before pydantic validation.

    Also caps generation: thinking is disabled and num_predict bounds the worst-case
    rambling outliers, so every judge call stays fast and cheap."""

    THINK = False  # probe-verified: thinking verdicts are hidden but still consume num_predict -> truncated/empty JSON; think=False returns clean JSON verdicts in 2-5s
    MAX_SCHEMA_ATTEMPTS = 5  # judges sometimes return empty/fenced/truncated JSON; bounded retries cover it
    DEFAULT_OPTIONS = {"num_predict": 4000}  # waffle + full JSON comfortably in one generation  # guards against runaway long-form answers

    def _options(self):
        merged = dict(self.DEFAULT_OPTIONS)
        merged.update(self.generation_kwargs or {})
        return merged

    @staticmethod
    def _extract_json(content: str) -> str:
        # strip in-content think tags some GLM variants emit
        content = re.sub(r"<think>.*?</think>", "", content, flags=re.S)
        m = re.search(r"```(?:json)?\s*(.*?)\s*```", content, re.S)
        text = m.group(1) if m else content
        for opener, closer in [("{", "}"), ("[", "]")]:
            start = text.find(opener)
            if start == -1:
                continue
            end = text.rfind(closer)
            if end > start:
                return text[start:end + 1]
        return text

    def generate(self, prompt: str, schema: Optional[BaseModel] = None) -> Tuple[Union[str, BaseModel], float]:
        chat_model = self.load_model()
        response = chat_model.chat(
            model=self.name,
            think=self.THINK,
            messages=[{"role": "user", "content": prompt}],
            format=schema.model_json_schema() if schema else None,
            options={"temperature": self.temperature, **self.generation_kwargs},
        )
        content = response["message"]["content"]
        if schema:
            if isinstance(content, list):
                content = "".join(part.get("text", "") for part in content if isinstance(part, dict))
            return schema.model_validate_json(self._extract_json(content)), 0
        return content, 0

    async def a_generate(self, prompt: str, schema: Optional[BaseModel] = None):
        import ollama
        chat_model = self.load_model(async_mode=True)
        # think=False keeps verdicts terse JSON; bounded schema retries cover empty/
        # truncated generations; the retry wrapper hard-caps each attempt so stalls
        # can never hang a MAX_CASE_WORKERS slot forever.
        chat_kwargs = dict(
            model=self.name,
            messages=[{"role": "user", "content": prompt}],
            format=schema.model_json_schema() if schema else None,
            think=self.THINK,
            options={"temperature": self.temperature, **self.generation_kwargs},
        )
        content, last_err = "", None
        for attempt in range(self.MAX_SCHEMA_ATTEMPTS):
            response: ollama.ChatResponse = await _achat_with_retries(chat_model, **chat_kwargs)
            content = response.message.content
            if isinstance(content, list):
                content = "".join(part.get("text", "") for part in content if isinstance(part, dict))
            if not schema:
                return content, 0
            try:
                return schema.model_validate_json(self._extract_json(content)), 0
            except (ValidationError, ValueError) as e:
                last_err = e
        raise RuntimeError(
            f"judge returned unparseable JSON in {self.MAX_SCHEMA_ATTEMPTS} attempts: "
            f"{last_err}; content={(content or '')[:160]!r}"
        )


if EVALUATOR_PROVIDER == "ollama":
    eval_model = CloudJudgeModel(
        model="minimax-m3.1-flash:cloud",   # try without ':cloud' if the ping 404s
        base_url="https://ollama.com",
    )
else:
    eval_model = OpenAIModel(model="gpt-5.4")  # uses OPENAI_API_KEY from .env -> api.openai.com

# Smoke test: fails loudly here, not mid-dataset, if the key/quota/model is bad.
# Retries a few times so transient DNS / network blips don't kill startup.
def _ping_with_retries(attempts=8, base_delay=4.0):
    class _Ping(BaseModel):
        ok: bool
        message: str

    last_exc = None
    for attempt in range(attempts):
        try:
            parsed, _ = eval_model.generate("Reply with JSON: {ok: true, message: \"ok\"}", schema=_Ping)
            assert parsed.ok, parsed
            return
        except Exception as err:
            last_exc = err
            if attempt < attempts - 1:
                delay = base_delay * (2 ** attempt) + 1
                print(f"[smoke retry] {type(err).__name__}: {str(err)[:100]} — retrying in {delay:.0f}s "
                      f"(attempt {attempt + 1}/{attempts})")
                time.sleep(delay)
    raise last_exc

_ping_with_retries()
print("evaluator ready:", eval_model.get_model_name())

evaluator ready: minimax-m3.1-flash:cloud (Ollama)


In [6]:
task_completion_metric = TaskCompletionMetric(threshold=0.5, model=eval_model, verbose_mode=True)
argument_correctness_metric = ArgumentCorrectnessMetric(threshold=0.5, model=eval_model, verbose_mode=True)
answer_relevancy_metric = AnswerRelevancyMetric(threshold=0.5, model=eval_model, verbose_mode=True)
hallucination_metric = HallucinationMetric(threshold=0.5, model=eval_model, verbose_mode=True)

metrics = [
    task_completion_metric,
    argument_correctness_metric,
    answer_relevancy_metric,
    hallucination_metric,
]

/tmp/ipykernel_2020802/3276406313.py:4: DeprecationWarning: 'HallucinationMetric' now scores in the same direction as every other deepeval metric: 1 is a pass, 0 is a failure, and 'threshold' is the MINIMUM passing score. It previously scored the proportion of violations, where 'threshold' was a maximum. Review any 'threshold' you pass and any code reading '.score' - a threshold of 0.2 that used to mean 'at most 20% violations' should now be 0.8. This notice will be removed in a future release.
  hallucination_metric = HallucinationMetric(threshold=0.5, model=eval_model, verbose_mode=True)


## Checkpointed Run Loop

Each finished (prompt, workflow) run is appended to `evaluation_results/raw_runs.jsonl`. On restart, finished runs are skipped, so a crash / credit outage never forces a full re-run.

In [7]:
import json, time

from tqdm import tqdm

CHECKPOINT_PATH = PROJECT_PATH / "evaluation_results" / "raw_runs.jsonl"
CHECKPOINT_PATH.parent.mkdir(exist_ok=True, parents=True)

def load_done_keys(path):
    done = set()
    if Path(path).exists():
        with open(path, encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    done.add((str(rec["prompt_id"]), rec["workflow"]))
                except (json.JSONDecodeError, KeyError):
                    continue  # tolerate a partially-written last line after a crash
    return done

done_keys = load_done_keys(CHECKPOINT_PATH)
pending = [
    (row, name, workflow)
    for row in prompt_rows
    for name, workflow in [("A", workflow_A), ("B", workflow_B), ("C", workflow_C)]
    if (str(row["prompt_id"]), name) not in done_keys
]
print(f"{len(done_keys)} checkpointed | {len(pending)} pending | {len(prompt_rows)*3} total")

for row, name, workflow in tqdm(pending, desc="workflow runs"):
    key = (str(row["prompt_id"]), name)

    _t0 = time.monotonic()
    report, tool_calls = await run_workflow(workflow, row["prompt"])
    run_seconds = round(time.monotonic() - _t0, 2)
    print(f"[{row['prompt_id']}/{name}] run_seconds={run_seconds}")

    record = {
        "prompt_id": row["prompt_id"],
        "category": row["category"],
        "stock": row["stock"],
        "prompt_style": row["prompt_style"],
        "prompt": row["prompt"],
        "workflow": name,
        "report": report,
        "tool_calls": tool_calls,
        "run_seconds": run_seconds,
    }
    with open(CHECKPOINT_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, default=str) + "\n")

print("run loop finished")

525 checkpointed | 0 pending | 525 total


workflow runs: 0it [00:00, ?it/s]

run loop finished


In [8]:
# Evaluate per case (not per batch of 10): each case is checkpointed to
# metric_results.jsonl right after it finishes, and per-metric failures are
# recorded instead of killing the run.

import json, time, random, asyncio
from IPython.display import clear_output
from tqdm import tqdm

records = []
with open(CHECKPOINT_PATH, encoding="utf-8") as f:
    for line in f:
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue

EVAL_CHECKPOINT = PROJECT_PATH / "evaluation_results" / "metric_results.jsonl"
evaluated = set()
if Path(EVAL_CHECKPOINT).exists():
    with open(EVAL_CHECKPOINT, encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
                evaluated.add((str(rec["prompt_id"]), rec["workflow"]))
            except (json.JSONDecodeError, KeyError):
                continue

pending = [rec for rec in records if (str(rec["prompt_id"]), rec["workflow"]) not in evaluated]
print(f"{len(evaluated)} evaluated | {len(pending)} pending | {len(records)} total")

MAX_CASE_WORKERS = 2   # verified: ollama cloud accepts 3 concurrent requests on this account
RELEVANCY_OUTPUT_CAP = 7000  # chars; uniform across all 525 cases so scores stay comparable

def make_metrics():
    """Fresh metric instances per case: deepeval metrics mutate state while measuring."""
    return [
        TaskCompletionMetric(threshold=0.5, model=eval_model, verbose_mode=True),
        ArgumentCorrectnessMetric(threshold=0.5, model=eval_model, verbose_mode=True),
        AnswerRelevancyMetric(threshold=0.5, model=eval_model, verbose_mode=True),
        HallucinationMetric(threshold=0.5, model=eval_model, verbose_mode=True),
    ]

async def evaluate_one(rec, sem, pbar):
    async with sem:
        tc = build_test_case(rec["prompt"], rec["report"], rec.get("tool_calls") or [])
        md_list = []
        for metric in make_metrics():
            try:
                tc_m = tc
                if isinstance(metric, AnswerRelevancyMetric):
                    # AnswerRelevancy makes one verdict-with-reason PER STATEMENT of the
                    # report, and report tails are mostly boilerplate ("Saved file: ...",
                    # follow-up offers) scoring borderline anyway. Uniform head-cap for
                    # all 525 cases keeps scores comparable while cutting tokens.
                    tc_m = LLMTestCase(
                        input=tc.input,
                        actual_output=(rec["report"] or "")[:RELEVANCY_OUTPUT_CAP],
                        context=tc.context,
                        tools_called=tc.tools_called,
                    )
                await metric.a_measure(tc_m, _show_indicator=False, _in_component=True)
                md_list.append({"name": metric.__class__.__name__,
                                "score": metric.score,
                                "success": metric.is_successful(),
                                "reason": getattr(metric, "reason", None)})
            except Exception as e:
                md_list.append({"name": metric.__class__.__name__,
                                "score": None, "success": None,
                                "reason": f"ERROR: {type(e).__name__}: {str(e)[:200]}"})

        ok_scores = [m["score"] is not None for m in md_list]
        out = {
            "prompt_id": rec["prompt_id"], "workflow": rec["workflow"],
            "success": all(ok_scores), "metrics": md_list,
        }
        with open(EVAL_CHECKPOINT, "a", encoding="utf-8") as f:
            f.write(json.dumps(out, default=str) + "\n")
        pbar.update(1)
        if pbar.n % 25 == 0:
            # keep the cell readable: drop interleaved verbose-log scroll,
            # the tqdm bar and the checkpoint file hold the real progress
            clear_output(wait=True)

async def evaluate_pending():
    sem = asyncio.Semaphore(MAX_CASE_WORKERS)
    pbar = tqdm(total=len(pending), desc="evaluating cases", unit="case")
    await asyncio.gather(*[evaluate_one(rec, sem, pbar) for rec in pending])
    pbar.close()

await evaluate_pending()

# per-workflow score summary
summary = defaultdict(list)
with open(EVAL_CHECKPOINT, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for md in rec["metrics"]:
            summary[(rec["workflow"], md["name"])].append(md["score"])

for (wf, metric_name), scores in sorted(summary.items()):
    scores = [s for s in scores if s is not None]
    if scores:
        print(f"workflow {wf} | {metric_name:30s} | avg: {sum(scores) / len(scores):.2f} | n: {len(scores)}")

525 evaluated | 0 pending | 525 total


evaluating cases: 0case [00:00, ?case/s]

workflow A | AnswerRelevancyMetric          | avg: 0.89 | n: 175
workflow A | ArgumentCorrectnessMetric      | avg: 0.91 | n: 175
workflow A | HallucinationMetric            | avg: 0.90 | n: 175
workflow A | TaskCompletionMetric           | avg: 0.58 | n: 175
workflow B | AnswerRelevancyMetric          | avg: 0.85 | n: 175
workflow B | ArgumentCorrectnessMetric      | avg: 0.89 | n: 175
workflow B | HallucinationMetric            | avg: 0.89 | n: 175
workflow B | TaskCompletionMetric           | avg: 0.51 | n: 175
workflow C | AnswerRelevancyMetric          | avg: 0.93 | n: 175
workflow C | ArgumentCorrectnessMetric      | avg: 0.93 | n: 175
workflow C | HallucinationMetric            | avg: 0.87 | n: 175
workflow C | TaskCompletionMetric           | avg: 0.53 | n: 175


## Retry failed metric measurements

`success` means "all four evaluators returned a score". Records whose metrics errored (score=None, reason=ERROR...) are re-measured here — only the missing metrics, so already-scored results are untouched. Reports/tool calls are reused from `raw_runs.jsonl`; no workflow re-runs. After the retry, the checkpoint file is rewritten atomically (dupe-free, same 525 keys) and the CSV is re-exported. A backup of the pre-retry checkpoint is kept as `metric_results.jsonl.bak-rerun`.

In [9]:
import asyncio, os, shutil, tempfile

from tqdm import tqdm

# self-contained constants: identical to cells defining them, so this section
# can also be run standalone after a fresh kernel without re-running the loops
CHECKPOINT_PATH = PROJECT_PATH / "evaluation_results" / "raw_runs.jsonl"
EVAL_CHECKPOINT = PROJECT_PATH / "evaluation_results" / "metric_results.jsonl"
MAX_CASE_WORKERS = 1
RELEVANCY_OUTPUT_CAP = 7000

# Re-measure only the metrics that errored (score=None) in the checkpoint.
# Keeps the already-good scores for the same (prompt_id, workflow) key, so a
# judge nondeterminism never perturbs existing results.

import tempfile

if EVAL_CHECKPOINT.exists():
    # dedupe last-wins: the file may contain retry re-appends for the same key
    recs_by_key = {}
    with open(EVAL_CHECKPOINT, encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            recs_by_key[(str(r["prompt_id"]), r["workflow"])] = r
    recs = list(recs_by_key.values())
else:
    recs = []
    print("no metric_results.jsonl yet - nothing to retry")

retry_map = {}   # index into recs -> metric class names to re-run
for idx, rec in enumerate(recs):
    missing = []
    for md in rec["metrics"]:
        if md["score"] is None or (md.get("reason") or "").startswith("ERROR"):
            missing.append(md["name"])
    if missing:
        retry_map[idx] = set(missing)

n_missing_calls = sum(len(v) for v in retry_map.values())
print(f"{len(retry_map)} records need retries ({n_missing_calls} metric measurements)")
for idx, names in sorted(retry_map.items()):
    r = recs[idx]
    print(f"  prompt {r['prompt_id']} / workflow {r['workflow']} -> {sorted(names)}")
assert n_missing_calls == 0 or retry_map, "sanity"

def make_metrics():
    # copied verbatim from the evaluate cell: fresh metric instances per case
    # (deepeval metrics mutate state while measuring), same threshold=0.5
    return [
        TaskCompletionMetric(threshold=0.5, model=eval_model, verbose_mode=True),
        ArgumentCorrectnessMetric(threshold=0.5, model=eval_model, verbose_mode=True),
        AnswerRelevancyMetric(threshold=0.5, model=eval_model, verbose_mode=True),
        HallucinationMetric(threshold=0.5, model=eval_model, verbose_mode=True),
    ]

METRIC_BY_NAME = {m.__class__.__name__: m.__class__ for m in metrics}

async def retry_one(idx, sem, pbar):
    rec = recs[idx]
    missing = retry_map[idx]
    async with sem:
        raw = raw_by_key[(str(rec["prompt_id"]), rec["workflow"])]
        tc = build_test_case(raw["prompt"], raw["report"], raw.get("tool_calls") or [])
        for new_metric in make_metrics():
            name = new_metric.__class__.__name__
            if name not in missing:
                continue
            try:
                tc_m = tc
                if isinstance(new_metric, AnswerRelevancyMetric):
                    tc_m = LLMTestCase(
                        input=tc.input,
                        actual_output=(raw["report"] or "")[:RELEVANCY_OUTPUT_CAP],
                        context=tc.context,
                        tools_called=tc.tools_called,
                    )
                await new_metric.a_measure(tc_m, _show_indicator=False, _in_component=True)
                result = {"name": name, "score": new_metric.score,
                          "success": new_metric.is_successful(),
                          "reason": getattr(new_metric, "reason", None)}
            except Exception as e:
                result = {"name": name, "score": None, "success": None,
                          "reason": f"ERROR: {type(e).__name__}: {str(e)[:200]}"}
            replaced = False
            for j, md in enumerate(rec["metrics"]):
                if md["name"] == name:
                    rec["metrics"][j] = result
                    replaced = True
                    break
            if not replaced:
                rec["metrics"].append(result)
        pbar.update(1)

# raw run records (identical to what evaluate_one originally measured)
raw_by_key = {}
if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
                raw_by_key[(str(r["prompt_id"]), r["workflow"])] = r
            except json.JSONDecodeError:
                continue
assert len(raw_by_key) == len(recs), "raw_runs.jsonl and metric_results.jsonl disagree"

if retry_map:
    # backup, then run retries
    backup = EVAL_CHECKPOINT.with_suffix(".jsonl.bak-rerun")
    shutil.copy2(EVAL_CHECKPOINT, backup)
    print(f"backup written: {backup}")

    sem = asyncio.Semaphore(MAX_CASE_WORKERS)
    pbar = tqdm(total=len(retry_map), desc="retrying cases", unit="case")
    await asyncio.gather(*[retry_one(idx, sem, pbar) for idx in retry_map])
    pbar.close()

    # recompute row-level success and rewrite the checkpoint atomically
    for rec in recs:
        rec["success"] = all(m["score"] is not None for m in rec["metrics"])

    fd, tmp_path = tempfile.mkstemp(dir=EVAL_CHECKPOINT.parent, suffix=".jsonl")
    with os.fdopen(fd, "w", encoding="utf-8") as f:
        for rec in recs:
            f.write(json.dumps(rec, default=str) + "\n")
    os.replace(tmp_path, EVAL_CHECKPOINT)

    still_bad = [ (r["prompt_id"], r["workflow"]) for r in recs
                  if not all(m["score"] is not None for m in r["metrics"]) ]
    print(f"checkpoint rewritten: {len(recs)} records")
    print(f"still failing after retry: {still_bad if still_bad else 'none'}")
else:
    print("nothing to retry")



0 records need retries (0 metric measurements)
nothing to retry


In [10]:
# Export final CSV from the two checkpoint files (works any time, no re-runs).

import csv
from datetime import datetime

def export_results_csv(raw_path=None, metrics_path=None, out_path=None) -> str:
    raw_path = raw_path or CHECKPOINT_PATH
    metrics_path = metrics_path or EVAL_CHECKPOINT
    out_path = out_path or PROJECT_PATH / "evaluation_results" / f"eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

    if not Path(metrics_path).exists() and not Path(raw_path).exists():
        print("no checkpoint data to export yet (fresh evaluation_results dir)")
        return None

    by_key = {}
    if Path(metrics_path).exists():
        with open(metrics_path, encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    by_key[(str(rec["prompt_id"]), rec["workflow"])] = rec
                except (json.JSONDecodeError, KeyError):
                    continue

    rows = []
    with open(raw_path, encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            out_row = {
                "prompt_id": rec["prompt_id"],
                "category": rec["category"],
                "stock": rec["stock"],
                "prompt_style": rec["prompt_style"],
                "prompt": rec["prompt"],
                "workflow": rec["workflow"],
                "run_seconds": rec.get("run_seconds"),  # None for legacy runs without timing
            }
            m = by_key.get((str(rec["prompt_id"]), rec["workflow"]), {})
            out_row["success"] = m.get("success")
            for md in m.get("metrics", []):
                out_row[f"{md['name']}_score"] = md["score"]
                out_row[f"{md['name']}_passed"] = md["success"]
                out_row[f"{md['name']}_reason"] = md["reason"]
            rows.append(out_row)

    if not rows:
        rows.append({k: None for k in
                     ["prompt_id", "category", "stock", "prompt_style", "prompt",
                      "workflow", "run_seconds", "success"]})
    Path(out_path).parent.mkdir(exist_ok=True, parents=True)
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    return str(out_path)

csv_path = export_results_csv()
print(f"results saved to {csv_path}" if csv_path
      else "no export produced (empty checkpoint dir)")

results saved to /run/media/horizontal_roterien_katze/U/project_finnessly/evaluation_results/eval_20261102_220917.csv
